In [0]:
import json
spark.sql("USE CATALOG football_catalog")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

def clean_league(bronze_table):
    raw_str = spark.table(f"bronze.{bronze_table}").collect()[0][0]
    data = json.loads(raw_str)
    
    # 1. Check if the data is nested under 'rounds' or top-level 'matches'
    if 'rounds' in data:
        # Standard nested structure
        matches_list = [m for r in data['rounds'] for m in r.get('matches', [])]
    elif 'matches' in data:
        # Simplified top-level structure
        matches_list = data['matches']
    else:
        # Fallback for empty or unknown data
        print(f"Warning: No match data found in {bronze_table}")
        return []

    # 2. Extract the core fields
    # We use .get() for the scores to prevent future crashes if a game is postponed
    return [
        {
            "team1": m.get('team1'), 
            "team2": m.get('team2'), 
            "s1": m.get('score', {}).get('ft', [0,0])[0], 
            "s2": m.get('score', {}).get('ft', [0,0])[1]
        } 
        for m in matches_list
    ]

# Process and Save
spark.createDataFrame(clean_league("raw_pl_2026")).write.mode("overwrite").saveAsTable("silver.pl_matches")
spark.createDataFrame(clean_league("raw_laliga_2026")).write.mode("overwrite").saveAsTable("silver.laliga_matches")

print("Silver: Matches cleaned successfully using adaptive parsing.")

Silver: Matches cleaned successfully using adaptive parsing.
